In [4]:
from langchain_unstructured import UnstructuredLoader

urls = [
    "https://www.victoriaonmove.com.au/local-removalists.html",
    "https://victoriaonmove.com.au/index.html",
    "https://victoriaonmove.com.au/contact.html",
]

docs = []

for url in urls:
    loader = UnstructuredLoader(web_url=url)
    docs.extend(loader.load())

print(f"Total documents: {len(docs)}")

Total documents: 318


In [5]:
docs

[Document(metadata={'category_depth': 0, 'languages': ['eng'], 'filetype': 'text/html', 'url': 'https://www.victoriaonmove.com.au/local-removalists.html', 'category': 'Title', 'element_id': '7085ab7563a6c4505727eb46b3585c2c'}, page_content='Local Removalists Melbourne'),
 Document(metadata={'languages': ['eng'], 'filetype': 'text/html', 'parent_id': '7085ab7563a6c4505727eb46b3585c2c', 'url': 'https://www.victoriaonmove.com.au/local-removalists.html', 'category': 'NarrativeText', 'element_id': '4e947dfb094d3844792dcfc13edcb7bd'}, page_content='Top-notch local moving services tailored to your needs — from studio apartments to large family homes.'),
 Document(metadata={'languages': ['eng'], 'filetype': 'text/html', 'parent_id': '7085ab7563a6c4505727eb46b3585c2c', 'url': 'https://www.victoriaonmove.com.au/local-removalists.html', 'category': 'UncategorizedText', 'element_id': '34946912f3647b759d52bfb30c052441'}, page_content='What We Do'),
 Document(metadata={'category_depth': 1, 'language

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# split data
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
data = text_splitter.split_documents(docs)


print("Total number of documents: ",len(data))

Total number of documents:  318


In [7]:
data[0]

Document(metadata={'category_depth': 0, 'languages': ['eng'], 'filetype': 'text/html', 'url': 'https://www.victoriaonmove.com.au/local-removalists.html', 'category': 'Title', 'element_id': '7085ab7563a6c4505727eb46b3585c2c'}, page_content='Local Removalists Melbourne')

In [9]:
data[1:3]

[Document(metadata={'languages': ['eng'], 'filetype': 'text/html', 'parent_id': '7085ab7563a6c4505727eb46b3585c2c', 'url': 'https://www.victoriaonmove.com.au/local-removalists.html', 'category': 'NarrativeText', 'element_id': '4e947dfb094d3844792dcfc13edcb7bd'}, page_content='Top-notch local moving services tailored to your needs — from studio apartments to large family homes.'),
 Document(metadata={'languages': ['eng'], 'filetype': 'text/html', 'parent_id': '7085ab7563a6c4505727eb46b3585c2c', 'url': 'https://www.victoriaonmove.com.au/local-removalists.html', 'category': 'UncategorizedText', 'element_id': '34946912f3647b759d52bfb30c052441'}, page_content='What We Do')]

In [4]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings # to convert the doc to numerical form
from langchain_openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

vectorstore = Chroma.from_documents(documents=data[1:2], embedding=OpenAIEmbeddings())

NameError: name 'data' is not defined

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
from dotenv import load_dotenv
load_dotenv()

embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-2-preview"
)

vectorstore = Chroma.from_documents(
    documents=data[1:3],
    embedding=embeddings,
    persist_directory="./vectorstore"
)

KeyboardInterrupt: 

In [1]:
from langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from dotenv import load_dotenv
load_dotenv()

embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-2-preview"
)

vectorstore = Chroma(
    persist_directory="./vectorstore",    # Your folder
    embedding_function=embeddings
)
vectorstore

In [2]:
print(vectorstore._collection.get())

{'ids': ['9a0fe0aa-1910-45c2-8b96-89f6ae58cfcc', '1fa6958e-9383-4f5e-ac03-018cd694fc19'], 'embeddings': None, 'documents': ['Top-notch local moving services tailored to your needs — from studio apartments to large family homes.', 'What We Do'], 'uris': None, 'included': ['metadatas', 'documents'], 'data': None, 'metadatas': [{'url': 'https://www.victoriaonmove.com.au/local-removalists.html', 'languages': ['eng'], 'category': 'NarrativeText', 'parent_id': '7085ab7563a6c4505727eb46b3585c2c', 'element_id': '4e947dfb094d3844792dcfc13edcb7bd', 'filetype': 'text/html'}, {'parent_id': '7085ab7563a6c4505727eb46b3585c2c', 'url': 'https://www.victoriaonmove.com.au/local-removalists.html', 'element_id': '34946912f3647b759d52bfb30c052441', 'languages': ['eng'], 'filetype': 'text/html', 'category': 'UncategorizedText'}]}


In [3]:
retriever = vectorstore.as_retriever()

In [4]:
# retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 6})

retrieved_docs = retriever.invoke("What kind of services they provide?")

In [27]:
# len(retrieved_docs)
print(retrieved_docs[1].page_content)

Top-notch local moving services tailored to your needs — from studio apartments to large family homes.


In [16]:
# from langchain.chains import create_retrieval_chain
# from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

In [23]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv

load_dotenv()
llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    temperature=0.4
)


In [29]:
chain = (
    # {
    #     "context": itemgetter("question") | retriever | RunnableLambda(format_docs),
    #     "question": itemgetter("question"),
    # }
    prompt
    | llm
    # | StrOutputParser()
)

response = chain.invoke(
    {
        "input": "What services do they provide?",
        "context": retrieved_docs
    }
)

print(response)

content=[{'type': 'text', 'text': 'Victoria On Move provides top-notch local moving services tailored to your specific needs. They handle a range of properties, from studio apartments to large family homes.', 'extras': {'signature': 'EjQKMgERTTIPzNtdyts1kZ2exE5CDj8g8etYKN0Y0kP68TD0jw4faOvVjErhRDk+h3BeuTe4'}}] additional_kwargs={} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--019f657c-3c13-7b81-bea9-dfead58e0288-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 405, 'output_tokens': 31, 'total_tokens': 436, 'input_token_details': {'cache_read': 0}}


In [30]:
print(response.content)

[{'type': 'text', 'text': 'Victoria On Move provides top-notch local moving services tailored to your specific needs. They handle a range of properties, from studio apartments to large family homes.', 'extras': {'signature': 'EjQKMgERTTIPzNtdyts1kZ2exE5CDj8g8etYKN0Y0kP68TD0jw4faOvVjErhRDk+h3BeuTe4'}}]


In [1]:
response = rag_chain.invoke({"input": "What kind of services they provide?"})
print(response["answer"])

NameError: name 'rag_chain' is not defined